# Phase 1 — Zhang et al. (ResNet18 + ConvGRU) reproduction (GIFGIF)

This notebook reproduces the **Phase 1 (paper-style)** setup:

- Use **7 emotions**: happiness, satisfaction (like), surprise, sadness, anger, disgust, fear  
- Use the **first 300 GIFs per class** (total 2100)  
- Split **80/20 per class** into train/test  
- Convert GIF → frames, then create **consecutive chunks of 4 frames** (`SEQ_LEN=4`)  
- If a GIF has <4 frames, **randomly pad** to 4 frames  
- If a GIF is long, split into **multiple 4-frame chunks**, each chunk becomes a training sample  
- Report **chunk-level** metrics (closest to the paper’s described protocol)

> ✅ Edit the **PATHS** cell below first.



In [11]:

# =========================
# PATHS (EDIT THESE)
# =========================
CSV_PATH = r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\gifgif_emotion_labels.csv"
# The CSV should have at least: gif_path, primary_emotion (or zhang_label), gif_id

# If your CSV already contains absolute paths in gif_path, keep GIF_ROOT = None
GIF_ROOT = None   # e.g. r"D:\IIT\Year 4\FYP\Datasets\GIFGIF_lucas\Data\gifgif-images-v1\gifgif-images"

# Training config (paper uses 100 epochs, batch 30, lr 1e-4, wd 1e-4)
EPOCHS = 100
BATCH  = 30
LR     = 1e-4
WD     = 1e-4
FREEZE_EPOCHS = 0   # optional stabilization; set to 0 to match "no freeze"
SEQ_LEN = 4
MAX_FRAMES_PER_GIF = 300   # safety cap; paper reports max ~303
SEED = 42

device = "cuda"  # set to "cpu" if needed


In [12]:

# =========================
# Imports
# =========================
import os, random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from PIL import Image, ImageSequence
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

from torchvision import models, transforms

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

from tqdm.auto import tqdm

# Repro
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(device if (device=="cuda" and torch.cuda.is_available()) else "cpu")
print("Device:", device)


Device: cpu


In [4]:

# =========================
# Load CSV and build labels
# =========================
Z_CLASSES = ["happiness", "satisfaction", "surprise", "sadness", "anger", "disgust", "fear"]

df = pd.read_csv(CSV_PATH)

# --- make gif_path absolute if needed ---
if GIF_ROOT is not None:
    import os
    def _join(p):
        if isinstance(p, str) and (os.path.isabs(p) or (len(p) > 2 and p[1] == ":")):
            return p
        return os.path.join(GIF_ROOT, str(p))
    df["gif_path"] = df["gif_path"].apply(_join)

# --- compute Zhang 7-class label from score columns ---
score_cols = [f"{c}_score" for c in Z_CLASSES]
missing = [c for c in score_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing score columns in CSV: {missing}")

# Ensure numeric
df[score_cols] = df[score_cols].apply(pd.to_numeric, errors="coerce")

# Check NaNs (should be none / minimal)
print("NaNs per score col:\n", df[score_cols].isna().sum())

# Argmax label across 7 scores
df["label"] = df[score_cols].idxmax(axis=1).str.replace("_score", "", regex=False)

df7 = df[df["label"].isin(Z_CLASSES)].copy()

print("Available counts (7-class):")
print(df7["label"].value_counts())
print("Total 7-class rows:", len(df7))



NaNs per score col:
 happiness_score       0
satisfaction_score    0
surprise_score        0
sadness_score         0
anger_score           0
disgust_score         0
fear_score            0
dtype: int64
Available counts (7-class):
label
happiness       2000
sadness          998
anger            911
surprise         679
satisfaction     652
disgust          459
fear             402
Name: count, dtype: int64
Total 7-class rows: 6101


In [5]:

# =========================
# Select "first 300" per class (paper protocol)
# =========================
parts = []
for lab in Z_CLASSES:
    sub = df7[df7["label"] == lab].copy()
    if len(sub) < 300:
        raise ValueError(f"Not enough samples for {lab}: {len(sub)} < 300")
    parts.append(sub.head(300))  # first 300, no shuffle

df2100 = pd.concat(parts, ignore_index=True)
print("Total selected:", len(df2100))
print(df2100["label"].value_counts())


Total selected: 2100
label
happiness       300
satisfaction    300
surprise        300
sadness         300
anger           300
disgust         300
fear            300
Name: count, dtype: int64


In [ ]:

# =========================
# Split 80/20 per class by GIF
# =========================
train_parts, test_parts = [], []
for lab in Z_CLASSES:
    sub = df2100[df2100["label"] == lab].copy()
    tr, te = train_test_split(sub, test_size=0.2, random_state=SEED, shuffle=True)
    train_parts.append(tr)
    test_parts.append(te)

train_df = pd.concat(train_parts, ignore_index=True).reset_index(drop=True)
test_df  = pd.concat(test_parts,  ignore_index=True).reset_index(drop=True)

print("Train:", len(train_df), "Test:", len(test_df))

label2id = {lab:i for i, lab in enumerate(Z_CLASSES)}
id2label = {i:lab for lab,i in label2id.items()}


Train: 1680 Test: 420


In [ ]:

# =========================
# Transforms
# =========================
TARGET_SIZE = (224, 224)

train_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

test_tf = transforms.Compose([
    transforms.Resize(TARGET_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])


In [ ]:

# =========================
# GIF → consecutive chunks of 4 frames
# =========================
rng = np.random.RandomState(SEED)

def load_gif_frames_full(path, max_frames=300):
    frames = []
    im = Image.open(path)
    for i, fr in enumerate(ImageSequence.Iterator(im)):
        if max_frames is not None and i >= max_frames:
            break
        frames.append(fr.convert("RGB"))
    return frames

def pad_random(frames, seq_len=4, rng=None):
    if rng is None:
        rng = np.random.RandomState(0)
    if len(frames) == 0:
        frames = [Image.new("RGB", TARGET_SIZE, (0,0,0))]
    out = frames[:]
    while len(out) < seq_len:
        out.append(out[rng.randint(0, len(out))])
    return out

def chunk_consecutive(frames, seq_len=4):
    chunks = []
    n = len(frames)
    if n < seq_len:
        chunks.append(pad_random(frames, seq_len=seq_len, rng=rng))
        return chunks
    for s in range(0, n, seq_len):
        clip = frames[s:s+seq_len]
        if len(clip) < seq_len:
            clip = pad_random(clip, seq_len=seq_len, rng=rng)
        chunks.append(clip)
    return chunks


In [ ]:

# =========================
# Dataset (expands each GIF into many chunks)
# =========================
class GifChunkDataset(Dataset):
    def __init__(self, df, transform, seq_len=4, max_frames=300):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.seq_len = seq_len
        self.max_frames = max_frames

        self.bad_gifs = 0
        self.index = []  # list of (row_idx, chunk_idx)

        for i in range(len(self.df)):
            path = self.df.loc[i, "gif_path"]
            try:
                frames = load_gif_frames_full(path, max_frames=self.max_frames)
                chunks = chunk_consecutive(frames, seq_len=self.seq_len)
                for c in range(len(chunks)):
                    self.index.append((i, c))
            except Exception:
                self.bad_gifs += 1
                self.index.append((i, 0))

        print(f"Built dataset: GIFs={len(self.df)}  Chunks={len(self.index)}  BadGIFs={self.bad_gifs}")

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        i, cidx = self.index[k]
        r = self.df.loc[i]
        lab = r["label"]
        path = r["gif_path"]

        try:
            frames = load_gif_frames_full(path, max_frames=self.max_frames)
            chunks = chunk_consecutive(frames, seq_len=self.seq_len)
            clip = chunks[min(cidx, len(chunks)-1)]
            x = torch.stack([self.transform(fr) for fr in clip], dim=0)  # [T,3,224,224]
        except Exception:
            x = torch.zeros((self.seq_len, 3, 224, 224), dtype=torch.float32)

        y = label2id[lab]
        return x, y


In [ ]:

# =========================
# DataLoaders (chunk-level)
# =========================
train_ds = GifChunkDataset(train_df, transform=train_tf, seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)
test_ds  = GifChunkDataset(test_df,  transform=test_tf,  seq_len=SEQ_LEN, max_frames=MAX_FRAMES_PER_GIF)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0, pin_memory=(device.type=="cuda"))
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=0, pin_memory=(device.type=="cuda"))

xb, yb = next(iter(train_loader))
print("Batch:", xb.shape, yb.shape)


Built dataset: GIFs=1680  Chunks=12725  BadGIFs=0
Built dataset: GIFs=420  Chunks=3370  BadGIFs=0
Batch: torch.Size([30, 4, 3, 224, 224]) torch.Size([30])


In [ ]:
print("Train chunks:", len(train_ds), "Test chunks:", len(test_ds))
print("Bad train GIFs:", train_ds.bad_gifs, "Bad test GIFs:", test_ds.bad_gifs)


Train chunks: 12725 Test chunks: 3370
Bad train GIFs: 0 Bad test GIFs: 0


In [ ]:

# =========================
# Model
# =========================
class ConvGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        padding = kernel_size // 2
        self.hidden_dim = hidden_dim
        self.conv_z = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_r = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)
        self.conv_h = nn.Conv2d(input_dim + hidden_dim, hidden_dim, kernel_size, padding=padding)

    def forward(self, x, h_prev):
        if h_prev is None:
            h_prev = torch.zeros(x.size(0), self.hidden_dim, x.size(2), x.size(3), device=x.device)
        cat = torch.cat([x, h_prev], dim=1)
        z = torch.sigmoid(self.conv_z(cat))
        r = torch.sigmoid(self.conv_r(cat))
        cat_r = torch.cat([x, r * h_prev], dim=1)
        h_hat = torch.tanh(self.conv_h(cat_r))
        h = (1 - z) * h_hat + z * h_prev
        return h

class ConvGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, kernel_size=3):
        super().__init__()
        self.cell = ConvGRUCell(input_dim, hidden_dim, kernel_size)

    def forward(self, x_seq):
        h = None
        for t in range(x_seq.size(1)):
            h = self.cell(x_seq[:, t], h)
        return h

class ResNetConvGRU(nn.Module):
    def __init__(self, num_classes=7, gru_hidden=256, dropout=0.4):
        super().__init__()
        base = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        self.backbone = nn.Sequential(*list(base.children())[:-2])  # [B,512,7,7]
        self.convgru = ConvGRU(input_dim=512, hidden_dim=gru_hidden, kernel_size=3)
        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.drop = nn.Dropout(p=dropout)
        self.fc = nn.Linear(gru_hidden, num_classes)

    def forward(self, x):
        feats = []
        for t in range(x.size(1)):
            feats.append(self.backbone(x[:, t]))
        feats = torch.stack(feats, dim=1)
        h = self.convgru(feats)
        v = self.pool(h).flatten(1)
        v = self.drop(v)
        return self.fc(v)

model = ResNetConvGRU(num_classes=len(Z_CLASSES), gru_hidden=256, dropout=0.4).to(device)
print("Params (M):", sum(p.numel() for p in model.parameters())/1e6)

def set_backbone_trainable(m, trainable: bool):
    for p in m.backbone.parameters():
        p.requires_grad = trainable

if FREEZE_EPOCHS > 0:
    set_backbone_trainable(model, False)
    print("Backbone frozen for", FREEZE_EPOCHS, "epochs.")


Params (M): 16.487495


In [ ]:

# =========================
# Train / Eval (chunk-level)
# =========================
def make_optimizer(model, frozen_phase: bool):
    if frozen_phase:
        return torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR, weight_decay=WD
        )
    return torch.optim.Adam([
        {"params": model.backbone.parameters(), "lr": 1e-5},
        {"params": model.convgru.parameters(),  "lr": 1e-4},
        {"params": model.fc.parameters(),       "lr": 1e-4},
    ], weight_decay=WD)

opt = make_optimizer(model, frozen_phase=(FREEZE_EPOCHS > 0))

def run_epoch(model, loader, train=True):
    model.train(train)
    losses = []
    y_true, y_pred = [], []

    for x, y in tqdm(loader, leave=False):
        x, y = x.to(device), y.to(device)
        if train:
            opt.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            if train:
                loss.backward()
                opt.step()

        losses.append(loss.item())
        y_true.extend(y.detach().cpu().numpy().tolist())
        y_pred.extend(logits.argmax(1).detach().cpu().numpy().tolist())

    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return float(np.mean(losses)), float(acc), float(p), float(r), float(f1)


In [ ]:

# =========================
# Training (saves best by ACC)
# =========================
history = []
best_acc = -1.0
CKPT_ACC_PATH = "phase1_zhang_best_acc.pt"

for epoch in range(1, EPOCHS + 1):

    if FREEZE_EPOCHS > 0 and epoch == FREEZE_EPOCHS + 1:
        set_backbone_trainable(model, True)
        opt = make_optimizer(model, frozen_phase=False)
        print(f"✅ Unfroze backbone at epoch {epoch}. Using differential LR (backbone=1e-5, head=1e-4).")

    tr_loss, tr_acc, tr_p, tr_r, tr_f1 = run_epoch(model, train_loader, train=True)
    te_loss, te_acc, te_p, te_r, te_f1 = run_epoch(model, test_loader,  train=False)

    row = dict(
        epoch=epoch,
        train_loss=tr_loss, train_acc=tr_acc, train_p=tr_p, train_r=tr_r, train_f1=tr_f1,
        test_loss=te_loss,  test_acc=te_acc,  test_p=te_p,  test_r=te_r,  test_f1=te_f1,
        bad_train_gifs=train_ds.bad_gifs, bad_test_gifs=test_ds.bad_gifs,
        train_chunks=len(train_ds), test_chunks=len(test_ds),
    )
    history.append(row)
    print(row)

    if te_acc > best_acc:
        best_acc = te_acc
        torch.save({"model": model.state_dict(), "epoch": epoch, "metrics": row}, CKPT_ACC_PATH)
        print("  💾 saved best ACC:", best_acc)

hist_df = pd.DataFrame(history)
hist_df.tail()


  0%|          | 0/425 [00:00<?, ?it/s]

  0%|          | 0/113 [00:00<?, ?it/s]

{'epoch': 1, 'train_loss': 1.4609606866275564, 'train_acc': 0.4580746561886051, 'train_p': 0.45919551381664603, 'train_r': 0.4552544069867634, 'train_f1': 0.454830413095468, 'test_loss': 2.341902761069019, 'test_acc': 0.23353115727002968, 'test_p': 0.2154735418122548, 'test_r': 0.23099894601802282, 'test_f1': 0.20279913215635761, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 12725, 'test_chunks': 3370}
  💾 saved best ACC: 0.23353115727002968


  0%|          | 0/425 [00:00<?, ?it/s]

  0%|          | 0/113 [00:00<?, ?it/s]

...

{'epoch': 7, 'train_loss': 0.059592091068899365, 'train_acc': 0.9767387033398821, 'train_p': 0.9768610460579737, 'train_r': 0.9767898798504391, 'train_f1': 0.9768233443180634, 'test_loss': 3.9740801965240884, 'test_acc': 0.2338278931750742, 'test_p': 0.23147915800575522, 'test_r': 0.23111954739194537, 'test_f1': 0.2234109393222119, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 12725, 'test_chunks': 3370}


  0%|          | 0/425 [00:00<?, ?it/s]

  0%|          | 0/113 [00:00<?, ?it/s]

{'epoch': 8, 'train_loss': 0.05818482831832679, 'train_acc': 0.9763457760314341, 'train_p': 0.9763493401613417, 'train_r': 0.9764711383612754, 'train_f1': 0.976403649201029, 'test_loss': 4.061345873944528, 'test_acc': 0.243026706231454, 'test_p': 0.22357122206945337, 'test_r': 0.2433347657233733, 'test_f1': 0.22484946321483212, 'bad_train_gifs': 0, 'bad_test_gifs': 0, 'train_chunks': 12725, 'test_chunks': 3370}


  0%|          | 0/425 [00:00<?, ?it/s]

In [ ]:

# =========================
# Final Evaluation (chunk-level)
# =========================
ckpt = torch.load(CKPT_ACC_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()
print("Loaded best epoch:", ckpt["epoch"])
print("Best snapshot:", ckpt["metrics"])

all_true, all_pred = [], []
with torch.no_grad():
    for x, y in tqdm(test_loader):
        x = x.to(device)
        logits = model(x)
        pred = logits.argmax(1).cpu().numpy().tolist()
        all_pred.extend(pred)
        all_true.extend(y.numpy().tolist())

print("Chunk-level Accuracy:", accuracy_score(all_true, all_pred))
print(classification_report(all_true, all_pred, target_names=Z_CLASSES, digits=4, zero_division=0))
print("Confusion Matrix:\n", confusion_matrix(all_true, all_pred))
